# Foundation models: CLIP and DINOv2 linear probes

Before you start: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom (about 30 minutes in total).

What this does:
1. Turns every image into a feature vector with a frozen model (CLIP ViT-L/14 and DINOv2 ViT-L/14). The models are not trained.
2. Trains a simple linear classifier (logistic regression) on the WikiArt **train** features, choosing its strength on WikiArt **val** only.
3. Scores it on WikiArt test, Met and Rijks, with the same metrics as the ResNet-50 baseline, plus seen/unseen artists and bootstrap intervals.
4. Bonus: CLIP **zero-shot** (no training at all, just the style names as text).

Uses the same `My Drive/colab_data.zip` as the training notebook. Results go to `My Drive/wikiart_runs/probes/`.

Note for the report: CLIP and DINOv2 were pre-trained on huge web image sets that may include these famous paintings, so their scores may be slightly flattering.

## 1. Connect Drive and unpack the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/data/final/wikiart_split.csv'):
    !cp /content/drive/MyDrive/colab_data.zip /content/
    !cd /content && unzip -q -o colab_data.zip && rm colab_data.zip
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls /content/data/final

## 2. Load the test sets (same split and museum test set as the baseline)

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
DATA = Path('/content/data')
OUT = Path('/content/drive/MyDrive/wikiart_runs/probes'); OUT.mkdir(parents=True, exist_ok=True)

split = pd.read_csv(DATA / 'final/wikiart_split.csv')
CLASSES = sorted(split['style'].unique())
museum = pd.read_csv(DATA / 'final/museum_test.csv')
museum = museum[museum['style'].isin(CLASSES)]

sets = {s: split[split['split'] == s] for s in ('train', 'val', 'test')}
sets['met'] = museum[museum['museum'] == 'met']
sets['rijks'] = museum[museum['museum'] == 'rijks']
for k in sets:
    df = sets[k]
    sets[k] = df[df['image'].map(lambda p: (DATA / p).exists())].reset_index(drop=True)
print('classes:', CLASSES)
for k, df in sets.items():
    print(f'{k:<6} {len(df):>6} images')
# Expected: train 18501, val 3786, test 3857, met 937, rijks 3092

## 3. Extract features (about 8 minutes per model; saved to Drive, so this runs only once)

In [ ]:
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, AutoModel, CLIPModel, CLIPProcessor

dev = 'cuda'

def vec(o):
    # newer transformers return an output object instead of a tensor
    return o if torch.is_tensor(o) else o.pooler_output
MODELS = {'clip': 'openai/clip-vit-large-patch14', 'dinov2': 'facebook/dinov2-large'}

class Images(Dataset):
    def __init__(self, paths, proc): self.paths, self.proc = paths, proc
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(DATA / self.paths[i]).convert('RGB')
        return self.proc(images=img, return_tensors='pt')['pixel_values'][0]

@torch.no_grad()
def extract(name):
    f = OUT / f'features_{name}.npz'
    if f.exists():
        print(name, 'features already on Drive'); return
    if name == 'clip':
        model = CLIPModel.from_pretrained(MODELS[name], torch_dtype=torch.float16).to(dev).eval()
        proc = CLIPProcessor.from_pretrained(MODELS[name]).image_processor
        embed = lambda x: vec(model.get_image_features(pixel_values=x))
    else:
        model = AutoModel.from_pretrained(MODELS[name], torch_dtype=torch.float16).to(dev).eval()
        proc = AutoImageProcessor.from_pretrained(MODELS[name])
        def embed(x):
            h = model(pixel_values=x).last_hidden_state
            return torch.cat([h[:, 0], h[:, 1:].mean(1)], 1)   # CLS + mean patch token, as in the DINOv2 paper
    feats = {}
    for k, df in sets.items():
        dl = DataLoader(Images(df['image'].tolist(), proc), batch_size=64, num_workers=2)
        feats[k] = np.concatenate([embed(x.to(dev, torch.float16)).float().cpu().numpy() for x in dl])
        print(f'{name} {k:<6} {feats[k].shape}', flush=True)
    np.savez(f, **feats)
    del model; torch.cuda.empty_cache()

for name in MODELS:
    extract(name)

## 4. Linear probes and scores

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score

def ece(y, probs, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e

def scores(y, probs):
    p = probs.argmax(1)
    top3 = (np.argsort(-probs, 1)[:, :3] == y[:, None]).any(1).mean()
    return dict(n=len(y), acc=accuracy_score(y, p), bal_acc=balanced_accuracy_score(y, p),
                macro_f1=f1_score(y, p, average='macro'), top3=top3, ece=ece(y, probs))

Y = {k: np.array([CLASSES.index(s) for s in df['style']]) for k, df in sets.items()}
probs_all, rows = {}, []
for name in MODELS:
    F = dict(np.load(OUT / f'features_{name}.npz'))
    F = {k: v / np.linalg.norm(v, axis=1, keepdims=True) for k, v in F.items()}
    sc = StandardScaler().fit(F['train'])
    F = {k: sc.transform(v) for k, v in F.items()}
    best = None
    for C in (0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0):          # tuned on WikiArt val only
        clf = LogisticRegression(C=C, max_iter=2000).fit(F['train'], Y['train'])
        b = balanced_accuracy_score(Y['val'], clf.predict(F['val']))
        print(f'{name}  C={C:<6} val bal acc {b:.3f}')
        if best is None or b > best[0]: best = (b, C, clf)
    print(f'{name}: chose C={best[1]}\n')
    for k in ('val', 'test', 'met', 'rijks'):
        pr = best[2].predict_proba(F[k]); probs_all[(name, k)] = pr
        rows.append(dict(model=f'{name} probe', set=k, **scores(Y[k], pr)))
        out = sets[k][['image', 'style']].copy(); out['pred'] = [CLASSES[i] for i in pr.argmax(1)]
        for i, c in enumerate(CLASSES): out[f'p_{c}'] = pr[:, i]
        if 'artist_group' in sets[k]: out['artist_group'] = sets[k]['artist_group'].values
        out.to_csv(OUT / f'preds_{name}_{k}.csv', index=False)
res = pd.DataFrame(rows)
print(res.round(3).to_string(index=False))

## 5. CLIP zero-shot (no training: compares each image with the text "a painting in the style of ...")

In [ ]:
@torch.no_grad()
def clip_text():
    model = CLIPModel.from_pretrained(MODELS['clip'], torch_dtype=torch.float16).to(dev).eval()
    proc = CLIPProcessor.from_pretrained(MODELS['clip'])
    names = [c.replace(' (Late Renaissance)', '') for c in CLASSES]
    templates = ['a painting in the style of {}.', 'a {} painting.', 'an artwork from the {} period.']
    T = []
    for n in names:
        t = proc(text=[tp.format(n) for tp in templates], return_tensors='pt', padding=True).to(dev)
        e = vec(model.get_text_features(**t)).float(); e = e / e.norm(dim=1, keepdim=True)
        T.append(e.mean(0))
    T = torch.stack(T); T = T / T.norm(dim=1, keepdim=True)
    del model; torch.cuda.empty_cache()
    return T.cpu().numpy()

T = clip_text()
raw = dict(np.load(OUT / 'features_clip.npz'))
for k in ('test', 'met', 'rijks'):
    f = raw[k] / np.linalg.norm(raw[k], axis=1, keepdims=True)
    logits = 100 * f @ T.T
    pr = np.exp(logits - logits.max(1, keepdims=True)); pr /= pr.sum(1, keepdims=True)
    probs_all[('clip0', k)] = pr
    rows.append(dict(model='clip zero-shot', set=k, **scores(Y[k], pr)))
res = pd.DataFrame(rows)
res.to_csv(OUT / 'probe_metrics.csv', index=False)
print(res[res['set'] != 'val'].round(3).to_string(index=False))

## 6. Seen vs unseen artists, museum gap, bootstrap intervals
Same comparison as the baseline: WikiArt test vs museum paintings by unseen artists.

In [ ]:
rng = np.random.default_rng(0)
B = 2000
def boot(y, p):
    n = len(y); return np.array([balanced_accuracy_score(y[i], p[i]) for i in (rng.integers(0, n, n) for _ in range(B))])

lines = []
for model, label in (('clip', 'CLIP probe'), ('dinov2', 'DINOv2 probe'), ('clip0', 'CLIP zero-shot')):
    pt = probs_all[(model, 'test')].argmax(1); bt = boot(Y['test'], pt)
    lines.append(f'\n{label}\n  wikiart_test        bal acc {100*balanced_accuracy_score(Y["test"], pt):5.1f} '
                 f'[{100*np.percentile(bt, 2.5):5.1f}, {100*np.percentile(bt, 97.5):5.1f}]')
    for k in ('met', 'rijks'):
        p = probs_all[(model, k)].argmax(1); g = sets[k]['artist_group'].values
        for grp in ('seen', 'unseen'):
            m = g == grp; b = boot(Y[k][m], p[m])
            lines.append(f'  {k:<6} {grp:<7} n={m.sum():>5}  bal acc {100*balanced_accuracy_score(Y[k][m], p[m]):5.1f} '
                         f'[{100*np.percentile(b, 2.5):5.1f}, {100*np.percentile(b, 97.5):5.1f}]')
            if grp == 'unseen':
                gap = bt - b; obs = balanced_accuracy_score(Y['test'], pt) - balanced_accuracy_score(Y[k][m], p[m])
                pval = min(1.0, 2 * min((gap <= 0).mean(), (gap >= 0).mean()))
                lines.append(f'         gap vs WikiArt test {100*obs:5.1f} points  95% CI [{100*np.percentile(gap, 2.5):5.1f}, '
                             f'{100*np.percentile(gap, 97.5):5.1f}]  p = {pval:.3f}')
report = '\n'.join(lines)
print(report)
(OUT / 'probe_report.txt').write_text(report)
print('\nSaved probe_metrics.csv, probe_report.txt and preds_*.csv to', OUT)

## 7. Per-style recall (paste this too)

In [ ]:
from sklearn.metrics import recall_score
for model in ('clip', 'dinov2'):
    print(f'\n{model} probe, recall per style')
    for k in ('test', 'met', 'rijks'):
        r = recall_score(Y[k], probs_all[(model, k)].argmax(1), average=None, labels=range(len(CLASSES)))
        print(f'  {k:<6}' + '  '.join(f'{c.split(" ")[0][:10]} {100*v:4.1f}' for c, v in zip(CLASSES, r)))